# Chapter 17: Paradigms of Spatial Simulation

*Part III — Foundations of Spatial Simulation*

## Learning Objectives

By the end of this chapter you will be able to:

- Distinguish the major paradigms for building discrete spatial models
- Recognize the tools that came before DisSModel and what each one got right
- Explain the gap in the Python ecosystem that motivated DisSModel

In [ ]:
# Standard imports — add chapter-specific imports below
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Part II ended with a full toolkit for asking *what is here* — vector geometry, raster grids, spatial weights, zonal statistics. This chapter opens a different question: *what happens next*. Simulating a system forward through time needs more than a snapshot of space; it needs a **paradigm** — a settled way of deciding what a "step" means, who or what changes at each step, and how those changes are described before they're ever coded. Before building anything, it's worth knowing the landscape of paradigms other tools have already staked out, since every one of them shaped what DisSModel became.

## What Is Discrete Spatial Modeling?

"Discrete" here means two things at once, and both matter. **Discrete time**: the model advances in fixed steps — ticks, generations, years — rather than continuously, the same `t, t+1, t+2, ...` structure every `Environment.run()` in this book's second half will use. **Discrete state**: at least part of what the model tracks takes on a finite set of values — a cell is `forest` or `burned`, not a continuously-varying real number — which is what separates this chapter's territory from Chapter 23's system dynamics, where stocks like temperature or population vary continuously even though time itself still advances in ticks.

A discrete spatial model adds one more ingredient on top: the state isn't a single global number, it's distributed across space — one value per cell, per parcel, per agent — and a big part of the model's logic is about how a value at one location affects a value somewhere nearby. That's the piece every paradigm below handles differently.

## Visual Data-Flow: Dinamica EGO

Dinamica EGO, developed at UFMG, takes modeling out of source code entirely: a model is a graph of connected functors, wired together on a canvas — a *Determine Transition Rates* box feeding a *Calculate Transitions* box feeding an *Update State* box — with parameters set through dialog boxes rather than function arguments. The appeal is real: a domain expert who has never written a line of code can read, and often build, a Dinamica model by following the arrows.

The trade-off is just as real. A visual graph is hard to diff in version control (Chapter 5's `git diff` on a `.dnp` project file is close to useless), hard to unit-test the way Chapter 5's `pytest` tested a plain Python function, and hard to compose with the rest of a research pipeline — a Dinamica model can't `import pandas` the way every model in this book's second half can. It solves accessibility at the direct cost of every reproducibility practice Chapter 5 and Chapter 21 built up.

## General-Purpose Engines: TerraME and LuccME

TerraME, built at INPE and developed further at institutions including this book's own research group, takes the opposite approach: a full scripting engine (embedding Lua) purpose-built for spatial simulation, with `CellularSpace`, `Agent`, `Society`, `Timer`, and `Event` as first-class language concepts rather than a library bolted onto a general-purpose language. Chapters 23 through 25 return to TerraME repeatedly and specifically, because DisSModel's own lineage runs directly through it — nearly every model in `dissmodel-sysdyn`, `dissmodel-ca`, and `dissmodel-abm` is a direct, documented port of an equivalent TerraME model, Lua rule compared line-by-line against Python rule.

LuccME, built on top of TerraME specifically for land-use and land-cover change modeling, adds a domain-specific layer above that: demand (how much of each land use is needed this period), allocation (where the change happens), and potential (which cells are suitable) as separate, swappable components. Chapter 26's `disslucc` packages inherit this same three-part structure directly — LuccME's architecture, not just its equations, survived the migration to Python.

## Agent-Based Frameworks: NetLogo and MASON

NetLogo, from Northwestern University, is the most widely taught agent-based modeling environment in the world — a Logo-derived language, a built-in visual interface, and a library of hundreds of example models (Wolf Sheep Predation among them, the direct ancestor of Chapter 25's `PredatorPreyModel`) that make it the default choice for a first course in ABM. Its cost is the mirror image of Dinamica's: NetLogo's own language doesn't interoperate with `numpy`, `pandas`, or `geopandas` at all — a NetLogo model lives in its own world, cut off from the rest of a Python-based research pipeline.

MASON, from George Mason University, sits at the other end of the same trade-off: a fast, general-purpose Java library for agent-based and discrete-event simulation, embeddable in a larger Java application, with no bundled domain assumptions the way TerraME assumes spatial cells or NetLogo assumes turtles-and-patches. Speed and flexibility come at the cost of everything NetLogo made easy — no visual interface out of the box, no beginner-friendly language, a steeper climb from zero to a first running model.

## The Python Gap

Line up the four tools above and a pattern falls out immediately: every one of them is either domain-specific and inaccessible to the Python scientific stack (Dinamica, TerraME/LuccME), or general-purpose and locked into its own language and runtime (NetLogo, MASON). None of them can `import geopandas` and hand a `GeoDataFrame` straight to a simulation the way Chapter 7 onward has made routine. A researcher choosing any of them accepts a hard boundary: data preparation happens in one tool, simulation happens in a completely different one, and every experiment pays a translation cost crossing that boundary.

That's the gap DisSModel exists to close — not a new paradigm invented from nothing, but TerraME's own paradigms (Chapters 23-25), reimplemented natively inside the Python ecosystem this entire book has been building fluency in since Chapter 1. Chapter 21 picks this exact thread back up.

## Exercises

1. **Match the paradigm to the problem.** For each scenario, name the tool from this chapter that fits best, and justify it in one sentence: (a) a domain expert with no programming background needs to build a land-use model, (b) a class of undergraduates is learning agent-based modeling for the first time, (c) a lab wants a spatial model that can read a `GeoDataFrame` and write results straight into a `pandas` analysis pipeline.
2. **Discrete time vs. discrete state.** Chapter 23's `Coffee` model advances in discrete ticks but its `temperature` stock is a continuous real number. Is `Coffee` a discrete spatial model by this chapter's definition? Why or why not?
3. **Trace one lineage.** Pick either `SchellingModel` (Chapter 25) or the `disslucc` demand/allocation/potential structure (Chapter 26) and, without looking ahead, write down which tool in this chapter you'd guess it descended from, based only on what this chapter described.
4. **The translation cost.** Describe, in your own words, what "a translation cost crossing that boundary" concretely means for a researcher moving data between, say, QGIS and NetLogo — what has to happen to the data, and what could go wrong in that step?

In [ ]:
# Your code here

## Summary

### Key concepts introduced

- Discrete spatial modeling: discrete time *and* discrete, spatially-distributed state, distinguishing this chapter's territory from Chapter 23's continuous system dynamics
- Dinamica EGO's visual data-flow paradigm, and its trade-off against version control and testing
- TerraME/LuccME's general-purpose spatial scripting engine, and LuccME's demand/allocation/potential structure that Chapter 26's `disslucc` inherits directly
- NetLogo and MASON as the two ends of the agent-based modeling spectrum — accessible-but-isolated versus fast-but-steep
- The Python gap: every tool above is either domain-specific and isolated from the scientific Python stack, or general-purpose and locked into its own language — the gap DisSModel exists to close

Chapter 18 starts filling that gap by hand — building Conway's Game of Life in pure Python, no framework at all, so that Chapter 21's DisSModel version has something real to be compared against.

## Further Reading

- Soares-Filho, B. S., Cerqueira, G. C., & Pennachin, C. L. (2002). "DINAMICA — a stochastic cellular automata model designed to simulate the landscape dynamics in an Amazonian colonization frontier." *Ecological Modelling*, 154(3), 217-235
- Carneiro, T. et al. LuccME: a framework for land-use change modeling. INPE.
- Wilensky, U. (1999). *NetLogo*. Center for Connected Learning and Computer-Based Modeling, Northwestern University: <https://ccl.northwestern.edu/netlogo/>
- Luke, S. et al. (2005). "MASON: A Multiagent Simulation Environment." *Simulation*, 81(7), 517-527
- TerraME on GitHub: <https://github.com/TerraME/terrame>